# EPOWER Energy Intelligence Demo

In this hands-on lab we build an end-to-end **Agentic AI application** for **EPOWER Energy** — a fictional German energy provider serving 20,000 residential and business customers. EPOWER pursues a **360° Energy Strategy** across six business domains:

| Strategy | Description |
|----------|-------------|
| **Supply** | Affordable electricity and gas tariffs |
| **Generate** | Solar installations for self-production |
| **Store** | Battery storage for energy independence |
| **Heat** | Heat pumps replacing fossil heating |
| **Drive** | E-mobility with wallboxes and charging tariffs |
| **Optimize** | Smart home technology for consumption reduction |

Together we'll create all database objects, load data across **6 business domains** (Sales, Billing, Service, VPP, HR, Finance), deploy dbt pipelines, and configure an **Intelligence Agent** that makes all enterprise data queryable through natural language.

**What you'll learn:** By the end of this lab you'll know how to make enterprise data AI-ready on Snowflake — from raw ingestion through governed data engineering to an intelligent agent that combines text-to-SQL (Semantic Views) with document retrieval (Cortex Search).

| Section | What we'll do |
|---------|-------------|
| **§1** Session Setup | Set up our role, Snowflake compute (virtual warehouse), and Snowpark session |
| **§2** Database & Schema Setup | Create the database and schemas (Bronze/Silver/Gold/Ops) |
| **§3** Synthetic Domain Data | Load Sales, Billing, Service, HR, Finance data — dim & fact tables (star schema) |
| **§4** Day-Ahead Electricity Prices | Connect to **real** EPEX day-ahead prices via external API (60-day backfill) |
| **§5** ePulse VPP Telemetry | Build the device registry + generate 60 days of **price-reactive** VPP telemetry |
| **§6** dbt Pipelines | Transform data through Bronze → Silver → Gold with native dbt |
| **§7** Daily Task Scheduling | Set up automated daily refresh: prices → telemetry → dbt |
| **§8** Semantic Views | Create 7 Semantic Views for text-to-SQL via Cortex Analyst |
| **§9** Document Parsing & Cortex Search | Parse PDFs/MDs and build 4 RAG search services |
| **§10** Intelligence Agent | Wire up a Cortex Agent that combines all capabilities |
| **§11** MCP Server | Expose everything via Model Context Protocol for external AI clients |
| **§12** Verification | Validate object counts and data |

**Runtime**: ~15 minutes | **Prerequisites**: See README.md (Git API integration required)

---
## Required Privileges

Three cells in this notebook require elevated privileges (§1, §2, §10). The table below lists the specific account-level privileges needed and the actions they enable. By default, all of these are inherited through the **ACCOUNTADMIN** role.

| Privilege | Actions Enabled | Section |
|-----------|----------------|---------|
| `CREATE ROLE` | Create `EPOWER_ROLE` | §1 |
| `CREATE WAREHOUSE` | Create `EPOWER_COMPUTE` warehouse | §1 |
| `MANAGE GRANTS` | Grant `CREATE DATABASE`, `EXECUTE TASK` on account to `EPOWER_ROLE`; grant role to user; grant usage on warehouse and integrations | §1, §2, §10 |
| `CREATE INTEGRATION` | Create external access integrations for API egress (`energy_charts_integration`, `ENERGY_EXTERNALACCESS`) | §2 |
| `CREATE SNOWFLAKE INTELLIGENCE` | Create the default Snowflake Intelligence object | §1 |
| `MODIFY` on Snowflake Intelligence object | Register the agent with Snowflake Intelligence | §10 |

> **If you cannot use ACCOUNTADMIN:** Ask your account administrator to grant these specific privileges to your role, or have them run the three marked cells (§1 setup, §2 integrations, §10 agent registration) on your behalf. All other cells run under `EPOWER_ROLE` with no elevated access.

---
## Architecture Overview

```
+-------------------------------------------------------------+
|                SNOWFLAKE INTELLIGENCE AGENT                  |
|            (12 tools: 7 Analyst + 4 Search + Chart)         |
+-------------------------------------------------------------+
                                  |
                 +----------------+------------------+
                 v                                   v
+--------------------------+          +--------------------------+
|     CORTEX ANALYST       |          |     CORTEX SEARCH        |
|      (Text-to-SQL)       |          |        (RAG)             |
|  7 Semantic Views        |          |  4 Document services     |
+--------------------------+          +--------------------------+
                 |                                   |
                 v                                   v
+--------------------------+          +--------------------------+
|      EPOWER_GOLD         |          |   Parsed Documents       |
|  Dims, Facts, VPP Marts  |          |   (PDF, Markdown)        |
+--------------------------+          +--------------------------+
                 ^
                 |
+--------------------------+
|    dbt Pipelines         |
|  Bronze -> Silver -> Gold |
|  (VPP + Market Data)     |
+--------------------------+
```

## 1. Session Setup

In [ ]:
# Import packages and get Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

### Role, Warehouse & Snowflake Intelligence Setup *(requires ACCOUNTADMIN)*

The next cell bootstraps the foundational infrastructure for the entire demo. It runs as `ACCOUNTADMIN` because these are account-level operations:

- **`EPOWER_ROLE`** — a dedicated role for this demo, granted to your current user and to `SYSADMIN`. It receives `CREATE DATABASE` and `EXECUTE TASK` privileges on the account.
- **`EPOWER_COMPUTE`** — a Gen 2 Small warehouse with 5-minute auto-suspend, used for all compute throughout the demo.
- **Snowflake Intelligence** — creates the default SI object so the agent can be registered with the Snowsight UI later (Section 10).

After setup, the session switches to `EPOWER_ROLE` for all remaining work.

&nbsp;

> **Note:** This is the first of three cells requiring `ACCOUNTADMIN`. If you don't have this role, ask your account administrator to run cells marked *(requires ACCOUNTADMIN)* before you begin.

In [ ]:
%%sql
-- Create role and warehouse
USE ROLE accountadmin;
CREATE SNOWFLAKE INTELLIGENCE IF NOT EXISTS snowflake_intelligence_object_default;
CREATE OR REPLACE ROLE EPOWER_ROLE;
SET current_user_name = CURRENT_USER();
GRANT ROLE EPOWER_ROLE TO USER IDENTIFIER($current_user_name);
GRANT ROLE EPOWER_ROLE TO ROLE SYSADMIN;
GRANT CREATE DATABASE ON ACCOUNT TO ROLE EPOWER_ROLE;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE EPOWER_ROLE;

CREATE OR REPLACE WAREHOUSE EPOWER_COMPUTE
    WAREHOUSE_SIZE = 'SMALL'
    GENERATION = '2'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE;
GRANT USAGE ON WAREHOUSE EPOWER_COMPUTE TO ROLE EPOWER_ROLE;

USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;

## 2. Database & Schema Setup

Let's start by creating the `EPOWER_DEMO` database with a **4-schema medallion architecture** that reflects the data lifecycle:

| Schema | Purpose | Contents |
|--------|---------|----------|
| **EPOWER_BRONZE** | Raw ingestion | Unprocessed API responses (day-ahead prices), raw IoT telemetry, device registry |
| **EPOWER_SILVER** | Cleaned & enriched | dbt staging models — flattened JSON, joined device-customer context |
| **EPOWER_GOLD** | Business-ready | Aggregated marts, dimension/fact tables, Semantic Views, Cortex Search services, Agent |
| **EPOWER_OPS** | Operational | Stored procedures, dbt project, stages, tasks |

&nbsp;

> **Snowflake Feature:** The medallion pattern (Bronze → Silver → Gold) separates raw, cleaned, and business-ready data. Combined with Snowflake's schema-level RBAC, this enables governed data access — analysts query Gold, engineers debug in Bronze.

In [ ]:
%%sql
-- Core database and schemas (Medallion Architecture)
CREATE OR REPLACE DATABASE EPOWER_DEMO;
USE DATABASE EPOWER_DEMO;

CREATE SCHEMA IF NOT EXISTS EPOWER_BRONZE COMMENT = 'Raw data: IoT telemetry, API ingestion';
CREATE SCHEMA IF NOT EXISTS EPOWER_SILVER COMMENT = 'Cleaned & enriched data (dbt staging)';
CREATE SCHEMA IF NOT EXISTS EPOWER_GOLD   COMMENT = 'Business-ready: dimensions, facts, metrics (dbt marts)';
CREATE SCHEMA IF NOT EXISTS EPOWER_OPS    COMMENT = 'Pipeline operations: stored procs, tasks, stages, network rules';

-- File format for CSV loading
CREATE OR REPLACE FILE FORMAT EPOWER_DEMO.EPOWER_OPS.CSV_FORMAT
    TYPE = 'CSV' FIELD_DELIMITER = ',' SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"' TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE NULL_IF = ('NULL', 'null', '', 'N/A');

### External Access Integrations *(requires ACCOUNTADMIN)*

The next cell creates **network rules** and **external access integrations** that allow stored procedures to call external APIs. This is needed for:

- **`energy_charts_integration`** — enables the day-ahead price fetch procedure (Section 4) to call the [Energy-Charts API](https://api.energy-charts.info)
- **`ENERGY_EXTERNALACCESS`** — enables broad HTTPS egress for the Snowflake Intelligence Agent (Section 10) to access web resources

Creating external access integrations is an account-level operation that requires the `ACCOUNTADMIN` role. The integrations are granted to `EPOWER_ROLE` so all subsequent steps run without elevated privileges.

&nbsp;

> **Note:** If you don't have access to the `ACCOUNTADMIN` role, ask your account administrator to run this cell for you before proceeding.

In [ ]:
%%sql
-- Network rules & external access integrations (bundled ACCOUNTADMIN section)
CREATE OR REPLACE NETWORK RULE EPOWER_DEMO.EPOWER_OPS.ENERGY_CHARTS_NETWORK_RULE
    MODE = EGRESS TYPE = HOST_PORT VALUE_LIST = ('api.energy-charts.info:443');

CREATE OR REPLACE NETWORK RULE EPOWER_DEMO.EPOWER_OPS.ENERGY_WEBACCESSRULE
    MODE = EGRESS TYPE = HOST_PORT VALUE_LIST = ('0.0.0.0:443');

USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION energy_charts_integration
    ALLOWED_NETWORK_RULES = (EPOWER_DEMO.EPOWER_OPS.ENERGY_CHARTS_NETWORK_RULE)
    ENABLED = true;
GRANT USAGE ON INTEGRATION energy_charts_integration TO ROLE EPOWER_ROLE;

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION ENERGY_EXTERNALACCESS
    ALLOWED_NETWORK_RULES = (EPOWER_DEMO.EPOWER_OPS.ENERGY_WEBACCESSRULE)
    ENABLED = true;
GRANT USAGE ON INTEGRATION ENERGY_EXTERNALACCESS TO ROLE EPOWER_ROLE;
USE ROLE EPOWER_ROLE;

## 3. Data Model & Loading

EPOWER's data consists of **3 data domains** that together enable cross-domain analytics:

| Domain | Schema | Key Tables |
|--------|--------|------------|
| **Business** (Sales, Billing, Service, HR, Finance) | `EPOWER_GOLD` | 13 dim tables + 10 fact tables |
| **Market** (Day-Ahead Electricity Prices) | `EPOWER_BRONZE` | `RAW_DAY_AHEAD_PRICES` (real EPEX Spot data) |
| **IoT** (ePulse Virtual Power Plant) | `EPOWER_BRONZE` | `EPULSE_DEVICES` + `RAW_EPULSE_IOT_TELEMETRY` |

**Loading strategy:**
- **Dimension tables** — static CSV files loaded from stage (Phase 1)
- **Fact tables** — generated dynamically by `POPULATE_FACT_TABLES()` with dates anchored to today (Phase 2)
- **Prices** — fetched from [energy-charts.info API](https://api.energy-charts.info/) (real market data)
- **Telemetry** — generated by `GENERATE_DAILY_TELEMETRY()` with cluster-specific energy profiles

> **Snowflake Feature:** Python Stored Procedures support `IMPORTS` — reference Python modules from a stage instead of inlining code. This keeps notebooks clean and logic maintainable.

In [ ]:
%%sql
-- ============================================================
-- COMPLETE DATA MODEL DDL
-- ============================================================
USE ROLE EPOWER_ROLE;
USE SCHEMA EPOWER_DEMO.EPOWER_OPS;

-- Internal stage for data and code modules
CREATE OR REPLACE STAGE EPOWER_STAGE FILE_FORMAT = CSV_FORMAT DIRECTORY = (ENABLE = TRUE);

-- === DIMENSION TABLES (EPOWER_GOLD) ===
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.product_category_dim (category_key INT PRIMARY KEY, category_name VARCHAR(100) NOT NULL, vertical VARCHAR(50) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.product_dim (product_key INT PRIMARY KEY, product_name VARCHAR(200) NOT NULL, category_key INT NOT NULL, category_name VARCHAR(100), vertical VARCHAR(50), capex_eur DECIMAL(12,2) DEFAULT 0, opex_eur_year DECIMAL(10,2) DEFAULT 0);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.customer_dim (customer_key INT PRIMARY KEY, customer_name VARCHAR(200) NOT NULL, customer_type VARCHAR(50), housing_type VARCHAR(100), vertical VARCHAR(50), address VARCHAR(200), city VARCHAR(100), zip VARCHAR(20), state VARCHAR(50), cluster_id VARCHAR(20), region_key INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.vpp_cluster_dim (cluster_id VARCHAR(20) PRIMARY KEY, cluster_name VARCHAR(100), centroid_lat FLOAT, centroid_lng FLOAT, region_character VARCHAR(200), compass_region VARCHAR(10), solar_multiplier FLOAT, solar_peak_kw FLOAT, hp_multiplier FLOAT, grid_bias FLOAT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.vendor_dim (vendor_key INT PRIMARY KEY, vendor_name VARCHAR(200) NOT NULL, vendor_type VARCHAR(100), vertical VARCHAR(50), address VARCHAR(200), city VARCHAR(100), state VARCHAR(50), zip VARCHAR(20));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.account_dim (account_key INT PRIMARY KEY, account_name VARCHAR(100) NOT NULL, account_type VARCHAR(50));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.department_dim (department_key INT PRIMARY KEY, department_name VARCHAR(100) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.region_dim (region_key INT PRIMARY KEY, region_name VARCHAR(100) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sales_rep_dim (sales_rep_key INT PRIMARY KEY, rep_name VARCHAR(200) NOT NULL, hire_date DATE);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.campaign_dim (campaign_key INT PRIMARY KEY, campaign_name VARCHAR(300) NOT NULL, objective VARCHAR(100));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.channel_dim (channel_key INT PRIMARY KEY, channel_name VARCHAR(100) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.employee_dim (employee_key INT PRIMARY KEY, employee_name VARCHAR(200) NOT NULL, gender VARCHAR(1), hire_date DATE);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.job_dim (job_key INT PRIMARY KEY, job_title VARCHAR(100) NOT NULL, job_level INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.location_dim (location_key INT PRIMARY KEY, location_name VARCHAR(200) NOT NULL);

-- === FACT TABLES (EPOWER_GOLD) ===
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sales_fact (sale_id INT PRIMARY KEY, date DATE NOT NULL, customer_key INT NOT NULL, product_key INT NOT NULL, sales_rep_key INT NOT NULL, region_key INT NOT NULL, vendor_key INT NOT NULL, amount DECIMAL(12,2) NOT NULL, units INT NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.billing_history (billing_id INT PRIMARY KEY, customer_key INT NOT NULL, billing_date DATE NOT NULL, billing_type VARCHAR(50) NOT NULL, consumption_kwh INT NOT NULL, amount DECIMAL(10,2) NOT NULL, payment_status VARCHAR(50));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.service_logs (log_id INT PRIMARY KEY, customer_key INT NOT NULL, log_date DATE NOT NULL, topic VARCHAR(100) NOT NULL, category VARCHAR(100), description VARCHAR(500), sentiment VARCHAR(50), channel VARCHAR(50), priority VARCHAR(50), resolution_date DATE, agent_key INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.finance_transactions (transaction_id INT PRIMARY KEY, date DATE NOT NULL, account_key INT NOT NULL, department_key INT NOT NULL, vendor_key INT NOT NULL, product_key INT NOT NULL, customer_key INT NOT NULL, amount DECIMAL(12,2) NOT NULL, approval_status VARCHAR(20), procurement_method VARCHAR(50), approver_id INT, approval_date DATE, purchase_order_number VARCHAR(50), contract_reference VARCHAR(100));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.marketing_campaign_fact (campaign_fact_id INT PRIMARY KEY, date DATE NOT NULL, campaign_key INT NOT NULL, product_key INT NOT NULL, channel_key INT NOT NULL, region_key INT NOT NULL, spend DECIMAL(10,2) NOT NULL, leads_generated INT NOT NULL, impressions INT NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.hr_employee_fact (hr_fact_id INT PRIMARY KEY, date DATE NOT NULL, employee_key INT NOT NULL, department_key INT NOT NULL, job_key INT NOT NULL, location_key INT NOT NULL, salary DECIMAL(10,2) NOT NULL, attrition_flag INT NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.customer_products (customer_product_id INT PRIMARY KEY, customer_key INT NOT NULL, product_key INT NOT NULL, category_key INT NOT NULL, category_name VARCHAR(100) NOT NULL, acquisition_date DATE NOT NULL, status VARCHAR(20) DEFAULT 'Active');
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sf_accounts (account_id VARCHAR(20) PRIMARY KEY, account_name VARCHAR(200) NOT NULL, customer_key INT NOT NULL, industry VARCHAR(100), vertical VARCHAR(50), billing_street VARCHAR(200), billing_city VARCHAR(100), billing_state VARCHAR(50), billing_postal_code VARCHAR(20), account_type VARCHAR(50), annual_revenue DECIMAL(15,2), employees INT, created_date DATE);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sf_opportunities (opportunity_id VARCHAR(20) PRIMARY KEY, sale_id INT, account_id VARCHAR(20) NOT NULL, opportunity_name VARCHAR(200) NOT NULL, stage_name VARCHAR(100) NOT NULL, amount DECIMAL(15,2) NOT NULL, probability DECIMAL(5,2), close_date DATE, created_date DATE, lead_source VARCHAR(100), type VARCHAR(100), campaign_id INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sf_contacts (contact_id VARCHAR(20) PRIMARY KEY, opportunity_id VARCHAR(20) NOT NULL, account_id VARCHAR(20) NOT NULL, first_name VARCHAR(100), last_name VARCHAR(100), email VARCHAR(200), phone VARCHAR(50), title VARCHAR(100), department VARCHAR(100), lead_source VARCHAR(100), campaign_no INT, created_date DATE);

-- === MARKET DATA (EPOWER_BRONZE) ===
USE SCHEMA EPOWER_BRONZE;
CREATE OR REPLACE TABLE RAW_DAY_AHEAD_PRICES (fetch_timestamp TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(), fetch_date DATE, raw_data VARIANT);

-- === VPP IoT (EPOWER_BRONZE) ===
CREATE OR REPLACE TABLE EPULSE_DEVICES (customer_key INT, gateway_id VARCHAR(20), has_solar BOOLEAN, has_battery BOOLEAN, has_heatpump BOOLEAN, is_vpp_enrolled BOOLEAN, enrollment_date DATE)
    COMMENT = 'Device registry: customers linked to ePulse IoT gateways with solar/battery/HP capabilities and VPP enrollment.';
CREATE OR REPLACE TABLE RAW_EPULSE_IOT_TELEMETRY (ts TIMESTAMP_NTZ, gateway_id VARCHAR(20), customer_key INT, solar_yield_kw FLOAT, battery_soc_pct FLOAT, heatpump_consumption_kw FLOAT, grid_import_export_kw FLOAT);

In [ ]:
# Upload workspace files to internal stage (CSVs, Python modules, documents)
import os

stage_name = '@EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE'
base_path = '../demo_data'

for folder in ['structured_data', 'unstructured_data', 'code']:
    local_path = f'{base_path}/{folder}'
    for root, dirs, files in os.walk(local_path):
        for file in files:
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(root, base_path)
            stage_path = f'{stage_name}/{rel_path}/'
            session.file.put(file_path, stage_path, auto_compress=False, overwrite=True)
            print(f'  {rel_path}/{file}')

print('\nFiles uploaded to stage')
session.sql('ALTER STAGE EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE REFRESH').collect()

In [ ]:
# Phase 1: Load dimension tables from stage (static reference data from CSVs)
dim_tables = [
    "product_category_dim", "product_dim", "customer_dim", "vpp_cluster_dim", "vendor_dim", "account_dim",
    "department_dim", "region_dim", "sales_rep_dim", "campaign_dim", "channel_dim",
    "employee_dim", "job_dim", "location_dim"
]

print("Loading dimension tables from stage...")
for t in dim_tables:
    try:
        r = session.sql(f"COPY INTO EPOWER_DEMO.EPOWER_GOLD.{t} FROM @EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/structured_data/{t}.csv FILE_FORMAT=EPOWER_DEMO.EPOWER_OPS.CSV_FORMAT ON_ERROR='CONTINUE'").collect()
        print(f"  {t}: {r[0]['rows_loaded']} rows")
    except: print(f"  {t}: skipped")
print("\nDimension tables loaded")

In [ ]:
%%sql
-- Phase 2: Register + run POPULATE_FACT_TABLES — generates 10 fact tables with dates anchored to today
-- Logic lives in demo_data/code/populate_fact_tables.py (uploaded to stage above)
CREATE OR REPLACE PROCEDURE EPOWER_DEMO.EPOWER_OPS.POPULATE_FACT_TABLES()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'pandas', 'numpy')
IMPORTS = ('@EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/code/populate_fact_tables.py')
HANDLER = 'populate_fact_tables.populate_fact_tables'
EXECUTE AS OWNER;

CALL EPOWER_DEMO.EPOWER_OPS.POPULATE_FACT_TABLES();

In [ ]:
%%sql
-- Verify data load: all tables should have rows
SELECT 'customer_dim' AS t, COUNT(*) AS row_count FROM EPOWER_DEMO.EPOWER_GOLD.customer_dim
UNION ALL SELECT 'vpp_cluster_dim', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.vpp_cluster_dim
UNION ALL SELECT 'sales_fact', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.sales_fact
UNION ALL SELECT 'billing_history', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.billing_history
UNION ALL SELECT 'service_logs', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.service_logs
UNION ALL SELECT 'customer_products', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.customer_products
ORDER BY t;

## 4. Day-Ahead Electricity Prices (Real Market Data)

We fetch **real day-ahead electricity prices** from the [Energy-Charts API](https://api.energy-charts.info/) (Fraunhofer ISE) for the DE-LU bidding zone (EPEX Spot). Prices are published daily around 13:00 CET and determine wholesale electricity costs for the next 24 hours in 15-minute intervals.

Two stored procedures (logic in `demo_data/code/day_ahead_prices.py`):
- `FETCH_DAY_AHEAD_PRICES(date)` — single day, idempotent
- `BACKFILL_DAY_AHEAD_PRICES()` — efficient bulk 60-day range call with CET delivery-day alignment

These prices drive the VPP telemetry simulation in Section 5 — batteries charge when prices are low and discharge when high.

In [ ]:
%%sql
-- Register price fetch procedures (logic in staged Python module)
CREATE OR REPLACE PROCEDURE EPOWER_DEMO.EPOWER_OPS.FETCH_DAY_AHEAD_PRICES(TARGET_DATE DATE)
RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests')
IMPORTS = ('@EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/code/day_ahead_prices.py')
HANDLER = 'day_ahead_prices.fetch_day_ahead_prices'
EXTERNAL_ACCESS_INTEGRATIONS = (energy_charts_integration) EXECUTE AS OWNER;

CREATE OR REPLACE PROCEDURE EPOWER_DEMO.EPOWER_OPS.BACKFILL_DAY_AHEAD_PRICES()
RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests')
IMPORTS = ('@EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/code/day_ahead_prices.py')
HANDLER = 'day_ahead_prices.backfill_day_ahead_prices'
EXTERNAL_ACCESS_INTEGRATIONS = (energy_charts_integration) EXECUTE AS OWNER;

In [ ]:
%%sql
-- Backfill 60 days + fetch today's prices
CALL EPOWER_DEMO.EPOWER_OPS.BACKFILL_DAY_AHEAD_PRICES();
CALL EPOWER_DEMO.EPOWER_OPS.FETCH_DAY_AHEAD_PRICES(CURRENT_DATE());

In [ ]:
%%sql
-- Verify price data
SELECT COUNT(*) AS total_days, MIN(fetch_date) AS earliest, MAX(fetch_date) AS latest
FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES;

## 5. ePulse VPP Telemetry Data

EPOWER operates an **ePulse Virtual Power Plant** — a network of distributed residential batteries coordinated to trade on the electricity market. The telemetry procedure below generates 60 days of hourly readings for ~4,500 battery devices, using the real day-ahead prices loaded in Section 4.

**Price-reactive battery strategy** (modeled after 1KOMMA5 Heartbeat AI):
| Price Zone | Battery Action | Grid Flow | SOC |
|-----------|---------------|-----------|-----|
| Negative (< 0) | Max Charge | Import +3.5-5 kW | 85-95% |
| Low (< P25) | Charge | Import +2-4.5 kW | 70-92% |
| Medium | Self-Consume | ~0 kW | 40-70% |
| High (> P75) | Discharge | Export -2 to -5 kW | 12-35% |

**Regional differentiation:** Solar yield, heat pump demand, and grid bias vary by VPP cluster using energy profile multipliers from `VPP_CLUSTER_DIM` (Freiburg: 1.35x solar, Hamburg: 0.75x).

In [ ]:
%%sql
-- Populate device registry from customer product ownership
USE SCHEMA EPOWER_DEMO.EPOWER_BRONZE;

INSERT INTO EPULSE_DEVICES
WITH hw AS (
    SELECT cp.customer_key,
        MAX(CASE WHEN cp.category_name='Solar & Storage' THEN 1 ELSE 0 END)=1 AS has_solar,
        MAX(CASE WHEN p.product_name LIKE '%Speicher%' THEN 1 ELSE 0 END)=1 AS has_battery,
        MAX(CASE WHEN cp.category_name='Heat Pumps' THEN 1 ELSE 0 END)=1 AS has_heatpump,
        MIN(cp.acquisition_date) AS first_acq
    FROM EPOWER_DEMO.EPOWER_GOLD.customer_products cp
    LEFT JOIN EPOWER_DEMO.EPOWER_GOLD.product_dim p ON cp.product_key=p.product_key
    WHERE cp.status='Active' GROUP BY cp.customer_key
)
SELECT customer_key, 'GW-'||LPAD(customer_key::VARCHAR,5,'0'),
    has_solar, has_battery, has_heatpump,
    CASE WHEN has_battery AND MOD(ABS(HASH(customer_key)),100)<90 THEN TRUE ELSE FALSE END,
    CASE WHEN has_battery AND MOD(ABS(HASH(customer_key)),100)<90 THEN DATEADD('day',30,first_acq) END
FROM hw WHERE has_solar OR has_battery OR has_heatpump;

SELECT COUNT(*) AS devices, SUM(CASE WHEN is_vpp_enrolled THEN 1 ELSE 0 END) AS vpp_enrolled FROM EPULSE_DEVICES;

In [ ]:
%%sql
-- VPP Telemetry Generation: price-reactive with regional energy profiles
-- Solar yield and HP consumption vary by cluster (JOIN VPP_CLUSTER_DIM)
-- Battery SOC and grid flow react to real EPEX spot prices

CREATE OR REPLACE PROCEDURE EPOWER_DEMO.EPOWER_OPS.GENERATE_DAILY_TELEMETRY()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
BEGIN
    LET days_generated INT := 0;
    LET max_ts TIMESTAMP_NTZ;
    LET start_date DATE;
    LET end_date DATE := CURRENT_DATE();

    SELECT MAX(ts)::DATE INTO :max_ts FROM EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY;

    IF (:max_ts IS NULL) THEN
        start_date := DATEADD(DAY, -59, :end_date);
    ELSE
        start_date := DATEADD(DAY, 1, :max_ts::DATE);
    END IF;

    IF (:start_date > :end_date) THEN
        RETURN 'Skipped: Telemetry already up to date (latest: ' || :max_ts::VARCHAR || ')';
    END IF;

    LET num_days INT := DATEDIFF(DAY, :start_date, :end_date) + 1;
    LET num_hours INT := :num_days * 24;

    INSERT INTO EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY
    WITH hours AS (
        SELECT DATEADD(HOUR, seq4(), :start_date::TIMESTAMP_NTZ) AS ts
        FROM TABLE(GENERATOR(ROWCOUNT => :num_hours))
    ),
    devices AS (
        SELECT d.customer_key, d.gateway_id, d.has_solar, d.has_heatpump, d.is_vpp_enrolled,
               COALESCE(cl.solar_multiplier, 1.0) AS solar_multiplier,
               COALESCE(cl.solar_peak_kw, 8.0) AS solar_peak_kw,
               COALESCE(cl.hp_multiplier, 1.0) AS hp_multiplier,
               COALESCE(cl.grid_bias, 0.0) AS grid_bias
        FROM EPOWER_BRONZE.EPULSE_DEVICES d
        INNER JOIN EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM c ON d.customer_key = c.customer_key
        LEFT JOIN EPOWER_DEMO.EPOWER_GOLD.VPP_CLUSTER_DIM cl ON c.cluster_id = cl.cluster_id
        WHERE d.has_battery
    ),
    price_stats AS (
        SELECT
            APPROX_PERCENTILE(r.raw_data:price[ts.INDEX]::FLOAT, 0.25) AS p25,
            APPROX_PERCENTILE(r.raw_data:price[ts.INDEX]::FLOAT, 0.75) AS p75
        FROM EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES r,
        LATERAL FLATTEN(input => r.raw_data:unix_seconds) ts
    ),
    hourly_prices AS (
        SELECT DATE_TRUNC('HOUR', TO_TIMESTAMP_NTZ(ts.VALUE::NUMBER)) AS price_hour,
               AVG(r.raw_data:price[ts.INDEX]::FLOAT) AS price_eur_mwh
        FROM EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES r,
        LATERAL FLATTEN(input => r.raw_data:unix_seconds) ts
        GROUP BY 1
    )
    SELECT
        h.ts,
        d.gateway_id,
        d.customer_key,
        -- Solar yield: scaled by cluster solar_multiplier and solar_peak_kw
        -- Using (low + ABS(RANDOM() % 1000000) / 1000000.0 * (high - low)) pattern since UNIFORM requires constants
        CASE WHEN d.has_solar AND HOUR(h.ts) BETWEEN 6 AND 20
             THEN ROUND(GREATEST(0, (0.5 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * (d.solar_peak_kw - 0.5)) * d.solar_multiplier * (1 - ABS(HOUR(h.ts)-13)*0.08)), 2)
             ELSE 0 END AS solar_yield_kw,
        -- Battery SOC: price-reactive (cluster-independent)
        CASE
            WHEN NOT d.is_vpp_enrolled THEN ROUND(20.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 60.0, 1)
            WHEN p.price_eur_mwh IS NULL THEN ROUND(30.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 40.0, 1)
            WHEN p.price_eur_mwh < 0           THEN ROUND(85.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 10.0, 1)
            WHEN p.price_eur_mwh < ps.p25      THEN ROUND(70.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 22.0, 1)
            WHEN p.price_eur_mwh > ps.p75      THEN ROUND(12.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 23.0, 1)
            ELSE ROUND(40.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 30.0, 1)
        END AS battery_soc_pct,
        -- Heat pump: scaled by cluster hp_multiplier
        CASE WHEN d.has_heatpump THEN ROUND((0.5 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 4.0) * d.hp_multiplier, 2) ELSE 0 END AS heatpump_consumption_kw,
        -- Grid: price-reactive + cluster grid_bias
        CASE
            WHEN NOT d.is_vpp_enrolled THEN ROUND((-2.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 7.0) + d.grid_bias, 2)
            WHEN p.price_eur_mwh IS NULL THEN ROUND((-1.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 4.0) + d.grid_bias, 2)
            WHEN p.price_eur_mwh < 0           THEN ROUND((3.5 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 1.5) + d.grid_bias, 2)
            WHEN p.price_eur_mwh < ps.p25      THEN ROUND((2.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 2.5) + d.grid_bias, 2)
            WHEN p.price_eur_mwh > ps.p75      THEN ROUND((-5.0 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 3.0) + d.grid_bias, 2)
            ELSE ROUND((-1.5 + ABS(MOD(RANDOM(), 1000000)) / 1000000.0 * 3.5) + d.grid_bias, 2)
        END AS grid_import_export_kw
    FROM devices d
    CROSS JOIN hours h
    CROSS JOIN price_stats ps
    LEFT JOIN hourly_prices p ON p.price_hour = DATE_TRUNC('HOUR', h.ts);

    days_generated := :num_days;
    RETURN 'Generated telemetry for ' || :days_generated || ' days (' || :start_date || ' to ' || :end_date || ')';
END
$$

In [ ]:
%%sql
-- Generate 60 days of telemetry using real day-ahead prices
CALL EPOWER_DEMO.EPOWER_OPS.GENERATE_DAILY_TELEMETRY();

-- Verify: ~60 days x ~4,500 battery devices x 24 hours
SELECT COUNT(*) AS telemetry_records,
       MIN(ts)::DATE AS earliest_date,
       MAX(ts)::DATE AS latest_date,
       DATEDIFF(DAY, MIN(ts), MAX(ts)) + 1 AS days_covered
FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY;

## 6. dbt Pipelines (Bronze → Silver → Gold)

Next, we transform the raw data from Sections 4 and 5 through a **medallion architecture** using dbt. It runs **natively inside Snowflake** — no local dbt CLI, no external CI/CD, and no separate compute infrastructure required.

### What is dbt on Snowflake?

[dbt (data build tool)](https://docs.getdbt.com/) is a transformation framework that lets data engineers write modular SQL models with built-in testing, documentation, and dependency management. Snowflake has a **first-class dbt integration** that goes beyond the traditional dbt CLI:

| Capability | Traditional dbt | dbt on Snowflake |
|-----------|----------------|-----------------|
| **Execution** | Runs on a local machine or CI server | Runs inside Snowflake as a managed object (`EXECUTE DBT PROJECT`) |
| **Deployment** | Deployed via git + CI/CD pipelines | Deployed as a Snowflake object (`CREATE DBT PROJECT`) synced from a Git workspace |
| **Scheduling** | Requires external orchestrator (Airflow, cron, dbt Cloud) | Native Snowflake Tasks — fully serverless scheduling |
| **Compute** | Relies on external machine for orchestration | Uses Snowflake compute (virtual warehouses) — scale up/down on demand |
| **Governance** | Separate access control | Inherits Snowflake RBAC — roles, grants, and audit trails |
| **Monitoring** | External logging / dbt Cloud | Snowflake query history, task history, and dbt artifacts stored in Snowflake |

The key SQL commands we'll use for native dbt are:
- **`CREATE DBT PROJECT`** — deploys a dbt project from a Git workspace or stage into a Snowflake schema
- **`EXECUTE DBT PROJECT ... ARGS = 'run'`** — materializes all models (equivalent to `dbt run`)
- **`EXECUTE DBT PROJECT ... ARGS = 'test'`** — runs data quality assertions (equivalent to `dbt test`)

### Pipeline Overview

The pipeline is **one interconnected flow**, not two isolated streams. Both telemetry and price data converge in `mart_vpp_price_optimization` to answer the core business question: **Is our Virtual Power Plant profitable?**

```
 EPOWER_BRONZE (Raw)              EPOWER_SILVER (Cleaned)            EPOWER_GOLD (Business-Ready)
 ===================              =====================              =============================

 EPULSE_DEVICES ----------> stg_devices (table) <--- customer_dim
                               |                      (EPOWER_GOLD)
                               | (enriched with
                               |  customer context)
                               v
 RAW_EPULSE_IOT_ ----> fct_epulse_telemetry ------> mart_vpp_capacity_hourly
 TELEMETRY               (table)                     (table: hourly fleet aggregation
                               |                       by region — solar, battery, grid)
                               |
                               |                      mart_vpp_price_optimization
                               |                      (table: THE JOIN — telemetry x prices)
                               \--------------------->  | price_zone (NEGATIVE/LOW/MEDIUM/HIGH)
                                                        | battery_action (CHARGE/DISCHARGE/...)
 RAW_DAY_AHEAD_ ------> stg_day_ahead_prices ------>  | import_cost_eur (grid buy cost)
 PRICES                  (incremental)          |      | export_revenue_eur (grid sell revenue)
                               |                |      | net_margin_eur (arbitrage profit)
                               |                |      | customer_margin_eur (70% share)
                               v                |      | epower_margin_eur (30% share)
                          mart_day_ahead_prices -+
                          (table: + EUR/kWh,
                           hour_of_day, day_of_week)
```

### Materialization Strategies

dbt supports different **materialization strategies** that control how models are stored in Snowflake. Let's look at the ones we use:

| Strategy | Used For | Behavior |
|----------|---------|----------|
| **table** | `stg_devices`, `fct_epulse_telemetry`, all marts | Full rebuild on every run — simple and deterministic |
| **incremental** | `stg_day_ahead_prices` | Only processes new/changed rows — efficient for append-heavy data like daily price feeds |
| **view** | (not used here, but common for light transformations) | No storage cost — recomputed on every query |

In our `dbt_project.yml`, staging models use `table` materialization (for telemetry) or `incremental` (for prices), while all Gold marts are `table` to ensure fast analytical query performance.

### Goal: Quantify VPP Arbitrage Value

Here's what each Gold model tells us:

| Gold Model | Purpose | Grain |
|-----------|---------|-------|
| `mart_vpp_capacity_hourly` | **Fleet operations** — how much solar/battery capacity is active? | Per hour × region |
| `mart_vpp_price_optimization` | **Financial analysis** — arbitrage margins from buying cheap / selling expensive | Per hour × customer |
| `mart_day_ahead_prices` | **Market analytics** — price patterns, trends, volatility | Per 15-min interval |

The revenue model splits arbitrage margins **70% customer / 30% EPOWER**, following industry-standard VPP aggregator economics.

### Step 6a: Run & Test in Snowsight (Interactive)

Before we deploy the dbt project as a Snowflake object, let's first run and test it interactively in the **Snowsight Workspace UI**:

1. Open the **Snowsight Workspace** (Projects → Workspaces → *your workspace*)
2. Navigate to the `epower_dbt/` folder
3. Click **Run** to execute all models — watch the DAG visualization, model status, and logs
4. Click **Test** to validate data quality assertions
5. Inspect the Silver and Gold tables in the schema browser to verify the output

This interactive step helps you understand the pipeline before we automate it.

### Step 6b: Deploy as Snowflake Object

Once you've validated it in the UI, we deploy the dbt project as a **managed Snowflake object** in `EPOWER_OPS` using `CREATE DBT PROJECT`. The project is synced from the Git-connected workspace — any future changes to the dbt models in the workspace can be re-deployed by re-running this statement. This makes the pipeline schedulable via Snowflake Tasks (Section 7).

The cell below **automatically detects your workspace name** and deploys the dbt project from it. It uses `SHOW WORKSPACES` to find the workspace containing `epower_dbt`, then constructs the `snow://workspace/...` URI dynamically — so it works regardless of what you named your workspace.

> **How it works**: The `FROM 'snow://workspace/USER$.PUBLIC."<name>"/versions/live/epower_dbt'` URI tells Snowflake to read the dbt project files directly from your live workspace. The `USER$` prefix is your personal workspace namespace.

In [ ]:
# Step 6b: Deploy dbt project — auto-detect workspace name

user = session.sql("SELECT CURRENT_USER()").collect()[0][0]
db = f"USER${user}"
workspaces = session.sql(f"SHOW WORKSPACES IN SCHEMA {db}.PUBLIC").collect()
workspace_name = None

for ws in workspaces:
    name = ws["name"]
    try:
        check = session.sql(
            f"""LIST 'snow://workspace/{db}.PUBLIC."{name}"/versions/live/epower_dbt/dbt_project.yml'"""
        ).collect()
        if len(check) > 0:
            workspace_name = name
            break
    except:
        continue

if workspace_name is None:
    raise RuntimeError(
        "Could not find a workspace containing 'epower_dbt/'. "
        "Please ensure your workspace has the epower_dbt folder at its root."
    )

workspace_uri = f'snow://workspace/{db}.PUBLIC."{workspace_name}"/versions/live/epower_dbt'
print(f"Detected workspace: {workspace_name}")
print(f"Deploying from:     {workspace_uri}")

deploy_sql = f"""CREATE OR REPLACE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT
    FROM '{workspace_uri}'"""
session.sql(deploy_sql).collect()
print("✓ dbt project deployed successfully")

### Step 6c: Scale Up Compute for Initial Pipeline Run

The first dbt run materializes all models from scratch — including `fct_epulse_telemetry` which processes ~9.7M rows and `mart_vpp_price_optimization` which joins telemetry with prices across millions of records. This needs significantly more compute than the daily incremental runs that follow.

**Snowflake Elastic Scalability** — this is one of Snowflake's most powerful features: we can resize compute on demand, with no downtime and no data migration:

- **Vertical Scaling** (scale up/down) — we increase the compute size for more powerful single-query performance. A LARGE virtual warehouse has 4× the compute resources of a SMALL, enabling complex joins and aggregations to complete faster. That's what we're doing here.
- **Horizontal Scaling** (scale out) — you can add multi-cluster capacity to handle more concurrent queries. Snowflake can automatically spin up additional clusters when query queuing is detected, distributing concurrent workloads across multiple identical compute clusters.

Both scaling dimensions take effect **within seconds** via a simple `ALTER WAREHOUSE` statement. After the pipeline completes, we'll scale back down to SMALL to minimize credit consumption — you only pay for what you use.

&nbsp;

> **Gen2 Compute** — this demo uses a Generation 2 virtual warehouse (`GENERATION = '2'`), which delivers improved performance through upgraded hardware and optimized software. Gen2 is particularly effective for DML-heavy workloads like the table materializations performed by dbt. See the [Snowflake Credit Consumption Table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf) for Gen2 credit rates by size.

In [ ]:
%%sql
ALTER WAREHOUSE EPOWER_COMPUTE SET WAREHOUSE_SIZE = 'LARGE';
SELECT 'Warehouse scaled up to LARGE for initial dbt pipeline run' AS status;

### Step 6d: Run & Test the Pipeline

Now let's run our pipeline! `EXECUTE DBT PROJECT ... ARGS = 'run'` materializes all models in dependency order — staging models first, then marts. This is equivalent to `dbt run` but executes entirely within Snowflake's compute infrastructure.

After that, we run `EXECUTE DBT PROJECT ... ARGS = 'test'` to validate data quality: not-null constraints, uniqueness checks, and referential integrity tests defined in the dbt schema YAML files.

In [ ]:
%%sql
-- Run the deployed dbt project to build all Silver/Gold models
EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT ARGS = 'run';

In [ ]:
%%sql
-- Test the deployed dbt project (data quality assertions)
EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT ARGS = 'test';

### Step 6e: Scale Down Compute

Pipeline's done — let's scale back to SMALL to reduce ongoing credit consumption. The daily incremental runs (via the scheduled Task in Section 7) only process new data and run efficiently on a Small compute cluster.

In [ ]:
%%sql
ALTER WAREHOUSE EPOWER_COMPUTE SET WAREHOUSE_SIZE = 'SMALL';
SELECT 'Warehouse scaled back to SMALL' AS status;

## 7. Daily Task Scheduling

Now that we've deployed the dbt project as a Snowflake object (`EPOWER_ANALYTICS_PROJECT`), let's schedule it via a **Snowflake Task**.

The task runs daily at **17:00 CET** (day-ahead prices are published around 13:00 CET) and executes three steps in sequence:

1. **Fetch next-day prices** — `FETCH_DAY_AHEAD_PRICES(CURRENT_DATE() + 1)`
2. **Generate telemetry** — `GENERATE_DAILY_TELEMETRY()` fills gaps up to today
3. **Run dbt** — `EXECUTE DBT PROJECT` refreshes the full Bronze → Silver → Gold pipeline

In [ ]:
%%sql
CREATE OR REPLACE TASK EPOWER_DEMO.EPOWER_OPS.TASK_DAILY_DATA_REFRESH
    WAREHOUSE = EPOWER_COMPUTE
    SCHEDULE = 'USING CRON 0 17 * * * Europe/Berlin'
    COMMENT = 'Daily 17:00 CET: fetch next-day prices, generate VPP telemetry, run dbt (Bronze → Silver → Gold)'
AS
    BEGIN
        CALL EPOWER_DEMO.EPOWER_OPS.FETCH_DAY_AHEAD_PRICES(CURRENT_DATE() + 1);
        CALL EPOWER_DEMO.EPOWER_OPS.GENERATE_DAILY_TELEMETRY();
        EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT 
            ARGS = 'run' 
            CONNECTION_NAME = 'default';
    END;

ALTER TASK EPOWER_DEMO.EPOWER_OPS.TASK_DAILY_DATA_REFRESH RESUME;
SHOW TASKS LIKE 'TASK_DAILY_DATA_REFRESH' IN SCHEMA EPOWER_DEMO.EPOWER_OPS;

## 8. Semantic Views for Cortex Analyst

Now we connect our data to the AI layer. **Semantic Views** define a business vocabulary over our Gold tables — specifying dimensions, metrics, relationships, and join paths in a way that Cortex Analyst can translate natural language questions into precise SQL.

We create one Semantic View per business domain, building a direct mapping between EPOWER's business model and the AI layer:

| Business Domain | Semantic View | What Users Can Ask |
|----------------|--------------|-------------------|
| Sales & Contracts | `ENERGY_SALES_SEMANTIC_VIEW` | Revenue by region, product mix, contract trends |
| Billing & Consumption | `BILLING_SEMANTIC_VIEW` | kWh patterns, billing amounts, payment history |
| Customer Energy | `CUSTOMER_ENERGY_SEMANTIC_VIEW` | Consumption by product ownership (solar, heat pump, etc.) |
| Customer Service | `SERVICE_SEMANTIC_VIEW` | Ticket volumes, sentiment, resolution times |
| HR & Workforce | `HR_SEMANTIC_VIEW` | Salaries, attrition, department headcount |
| Market Prices | `MARKET_PRICES_SEMANTIC_VIEW` | Day-ahead price trends, volatility, patterns |
| VPP Telemetry | `EPULSE_VPP_SEMANTIC_VIEW` | Solar yield, battery SOC, grid flow by device/region |

&nbsp;

> **Snowflake Feature:** Semantic Views are the bridge between governed data and natural language access. They ensure that Cortex Analyst generates correct, performant SQL — grounded in actual table structures and business rules — rather than guessing at schema relationships.

### Cortex Search Services for High-Cardinality Columns

Cortex Analyst needs exact literal values to generate correct SQL filters (e.g. `WHERE customer_name = 'Max Müller'`). For high-cardinality columns with many distinct values, we create **Cortex Search Services** that enable fuzzy semantic matching between the user's natural language input and the actual database values.

Each service indexes the distinct values of a column and is referenced in the semantic view dimensions via `WITH CORTEX SEARCH SERVICE`. For example, when a user asks *"Show contracts for Mueller"*, the search service resolves this to the exact value `Max Müller` before SQL generation.

| Search Service | Column | Source Table | Semantic Views Using It |
|---|---|---|---|
| `_CA_CUSTOMER_NAME` | `customer_name` | `customer_dim` | Sales, Billing, Service, Customer Energy, VPP |
| `_CA_CITY` | `city` | `customer_dim` | Sales, VPP |
| `_CA_PRODUCT_NAME` | `product_name` | `product_dim` | Sales |
| `_CA_VENDOR_NAME` | `vendor_name` | `vendor_dim` | Sales |
| `_CA_REP_NAME` | `rep_name` | `sales_rep_dim` | Sales |

In [ ]:
%%sql
-- Cortex Search Services for High-Cardinality Columns (Dynamic Literal Retrieval)
CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME
  ON customer_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT customer_name FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CITY
  ON city
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT city FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_PRODUCT_NAME
  ON product_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT product_name FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_VENDOR_NAME
  ON vendor_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT vendor_name FROM EPOWER_DEMO.EPOWER_GOLD.VENDOR_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_REP_NAME
  ON rep_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT rep_name FROM EPOWER_DEMO.EPOWER_GOLD.SALES_REP_DIM);

In [ ]:
%%sql
-- Energy Sales Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY) WITH SYNONYMS = ('Kunden','customers'),
        PRODUCTS AS EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM PRIMARY KEY (PRODUCT_KEY) WITH SYNONYMS = ('Produkte','products'),
        REGIONS AS EPOWER_DEMO.EPOWER_GOLD.REGION_DIM PRIMARY KEY (REGION_KEY) WITH SYNONYMS = ('Regionen','regions'),
        CONTRACTS AS EPOWER_DEMO.EPOWER_GOLD.SALES_FACT PRIMARY KEY (SALE_ID) WITH SYNONYMS = ('Vertraege','contracts'),
        SALES_REPS AS EPOWER_DEMO.EPOWER_GOLD.SALES_REP_DIM PRIMARY KEY (SALES_REP_KEY) WITH SYNONYMS = ('Berater','consultants'),
        VENDORS AS EPOWER_DEMO.EPOWER_GOLD.VENDOR_DIM PRIMARY KEY (VENDOR_KEY) WITH SYNONYMS = ('Lieferanten','vendors','partners')
    )
    RELATIONSHIPS (
        CONTRACTS_TO_CUSTOMERS AS CONTRACTS(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY),
        CONTRACTS_TO_PRODUCTS AS CONTRACTS(PRODUCT_KEY) REFERENCES PRODUCTS(PRODUCT_KEY),
        CONTRACTS_TO_REGIONS AS CONTRACTS(REGION_KEY) REFERENCES REGIONS(REGION_KEY),
        CONTRACTS_TO_REPS AS CONTRACTS(SALES_REP_KEY) REFERENCES SALES_REPS(SALES_REP_KEY),
        CONTRACTS_TO_VENDORS AS CONTRACTS(VENDOR_KEY) REFERENCES VENDORS(VENDOR_KEY)
    )
    FACTS (
        CONTRACTS.CONTRACT_AMOUNT AS AMOUNT,
        CONTRACTS.CONTRACT_UNITS AS UNITS,
        PRODUCTS.CAPEX_EUR AS CAPEX_EUR,
        PRODUCTS.OPEX_EUR_YEAR AS OPEX_EUR_YEAR
    )
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        CUSTOMERS.CUSTOMER_TYPE AS CUSTOMER_TYPE,
        CUSTOMERS.CITY AS CITY WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CITY,
        PRODUCTS.PRODUCT_KEY AS PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME AS PRODUCT_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_PRODUCT_NAME,
        PRODUCTS.CATEGORY_NAME AS CATEGORY_NAME,
        REGIONS.REGION_KEY AS REGION_KEY,
        REGIONS.REGION_NAME AS REGION_NAME,
        CONTRACTS.SALE_ID AS SALE_ID,
        CONTRACTS.CONTRACT_DATE AS DATE,
        SALES_REPS.REP_NAME AS REP_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_REP_NAME,
        VENDORS.VENDOR_NAME AS VENDOR_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_VENDOR_NAME
    )
    METRICS (
        CONTRACTS.TOTAL_REVENUE AS SUM(CONTRACTS.CONTRACT_AMOUNT),
        CONTRACTS.TOTAL_CONTRACTS AS COUNT(CONTRACTS.SALE_ID),
        CONTRACTS.AVG_CONTRACT_VALUE AS AVG(CONTRACTS.CONTRACT_AMOUNT),
        PRODUCTS.AVG_CAPEX AS AVG(PRODUCTS.CAPEX_EUR),
        PRODUCTS.AVG_OPEX AS AVG(PRODUCTS.OPEX_EUR_YEAR)
    );

In [ ]:
%%sql
-- Billing Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
        BILLING AS EPOWER_DEMO.EPOWER_GOLD.BILLING_HISTORY PRIMARY KEY (BILLING_ID) WITH SYNONYMS = ('Rechnungen','invoices')
    )
    RELATIONSHIPS (BILLING_TO_CUSTOMERS AS BILLING(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY))
    FACTS (
        BILLING.CONSUMPTION AS CONSUMPTION_KWH,
        BILLING.BILLING_AMOUNT AS AMOUNT
    )
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        CUSTOMERS.HOUSING_TYPE AS HOUSING_TYPE,
        BILLING.BILLING_ID AS BILLING_ID,
        BILLING.BILLING_DATE AS BILLING_DATE,
        BILLING.BILLING_TYPE AS BILLING_TYPE,
        BILLING.PAYMENT_STATUS AS PAYMENT_STATUS
    )
    METRICS (
        BILLING.TOTAL_CONSUMPTION AS SUM(BILLING.CONSUMPTION),
        BILLING.AVG_CONSUMPTION AS AVG(BILLING.CONSUMPTION)
    );

-- Service Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
        TICKETS AS EPOWER_DEMO.EPOWER_GOLD.SERVICE_LOGS PRIMARY KEY (LOG_ID) WITH SYNONYMS = ('Tickets','Anfragen')
    )
    RELATIONSHIPS (TICKETS_TO_CUSTOMERS AS TICKETS(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY))
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        TICKETS.LOG_ID AS LOG_ID,
        TICKETS.LOG_DATE AS LOG_DATE,
        TICKETS.TOPIC AS TOPIC,
        TICKETS.CATEGORY AS CATEGORY,
        TICKETS.SENTIMENT AS SENTIMENT,
        TICKETS.PRIORITY AS PRIORITY
    )
    METRICS (
        TICKETS.TOTAL_TICKETS AS COUNT(TICKETS.LOG_ID)
    );

-- Customer Energy Semantic View (consumption + product ownership + VPP enrollment)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
        BILLING AS EPOWER_DEMO.EPOWER_GOLD.BILLING_HISTORY PRIMARY KEY (BILLING_ID),
        OWNERSHIP AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_PRODUCTS PRIMARY KEY (CUSTOMER_PRODUCT_ID),
        DEVICES AS EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES PRIMARY KEY (CUSTOMER_KEY) WITH SYNONYMS = ('VPP','Virtual Power Plant','Geraete')
    )
    RELATIONSHIPS (
        BILLING_TO_CUSTOMERS AS BILLING(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY),
        OWNERSHIP_TO_CUSTOMERS AS OWNERSHIP(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY),
        DEVICES_TO_CUSTOMERS AS DEVICES(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY)
    )
    FACTS (
        BILLING.CONSUMPTION AS CONSUMPTION_KWH,
        BILLING.BILLING_AMOUNT AS AMOUNT
    )
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        CUSTOMERS.HOUSING_TYPE AS HOUSING_TYPE,
        BILLING.BILLING_DATE AS BILLING_DATE,
        BILLING.BILLING_TYPE AS BILLING_TYPE,
        OWNERSHIP.CATEGORY_NAME AS CATEGORY_NAME,
        DEVICES.GATEWAY_ID AS GATEWAY_ID,
        DEVICES.HAS_SOLAR AS HAS_SOLAR,
        DEVICES.HAS_BATTERY AS HAS_BATTERY,
        DEVICES.HAS_HEATPUMP AS HAS_HEATPUMP,
        DEVICES.IS_VPP_ENROLLED AS IS_VPP_ENROLLED,
        DEVICES.ENROLLMENT_DATE AS ENROLLMENT_DATE
    )
    METRICS (
        BILLING.AVG_CONSUMPTION AS AVG(BILLING.CONSUMPTION)
    );

-- HR Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW
    TABLES (
        DEPARTMENTS AS EPOWER_DEMO.EPOWER_GOLD.DEPARTMENT_DIM PRIMARY KEY (DEPARTMENT_KEY),
        EMPLOYEES AS EPOWER_DEMO.EPOWER_GOLD.EMPLOYEE_DIM PRIMARY KEY (EMPLOYEE_KEY),
        HR_RECORDS AS EPOWER_DEMO.EPOWER_GOLD.HR_EMPLOYEE_FACT PRIMARY KEY (HR_FACT_ID),
        JOBS AS EPOWER_DEMO.EPOWER_GOLD.JOB_DIM PRIMARY KEY (JOB_KEY)
    )
    RELATIONSHIPS (
        HR_TO_DEPARTMENTS AS HR_RECORDS(DEPARTMENT_KEY) REFERENCES DEPARTMENTS(DEPARTMENT_KEY),
        HR_TO_EMPLOYEES AS HR_RECORDS(EMPLOYEE_KEY) REFERENCES EMPLOYEES(EMPLOYEE_KEY),
        HR_TO_JOBS AS HR_RECORDS(JOB_KEY) REFERENCES JOBS(JOB_KEY)
    )
    FACTS (
        HR_RECORDS.ATTRITION_FLAG AS ATTRITION_FLAG,
        HR_RECORDS.EMPLOYEE_SALARY AS SALARY
    )
    DIMENSIONS (
        DEPARTMENTS.DEPARTMENT_NAME AS DEPARTMENT_NAME,
        EMPLOYEES.EMPLOYEE_NAME AS EMPLOYEE_NAME,
        EMPLOYEES.GENDER AS GENDER,
        JOBS.JOB_TITLE AS JOB_TITLE
    )
    METRICS (
        HR_RECORDS.TOTAL_SALARY AS SUM(HR_RECORDS.EMPLOYEE_SALARY),
        HR_RECORDS.AVG_SALARY AS AVG(HR_RECORDS.EMPLOYEE_SALARY)
    );

In [ ]:
%%sql
-- ePulse Day-Ahead Prices Semantic View (Energy Market)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW
    TABLES (
        PRICES AS EPOWER_DEMO.EPOWER_SILVER.STG_DAY_AHEAD_PRICES PRIMARY KEY (START_TIME)
            WITH SYNONYMS = ('Strompreise','electricity prices','day-ahead','Boersenpreise','spot prices','Marktpreise','energy market prices')
    )
    FACTS (
        PRICES.PRICE_EUR_MWH AS PRICE_EUR_MWH
    )
    DIMENSIONS (
        PRICES.START_TIME AS START_TIME,
        PRICES.END_TIME AS END_TIME,
        PRICES.FETCH_DATE AS FETCH_DATE
    )
    METRICS (
        PRICES.AVG_PRICE AS AVG(PRICES.PRICE_EUR_MWH),
        PRICES.MAX_PRICE AS MAX(PRICES.PRICE_EUR_MWH),
        PRICES.MIN_PRICE AS MIN(PRICES.PRICE_EUR_MWH)
    );

In [ ]:
%%sql
-- ePulse VPP Telemetry Semantic View (Virtual Power Plant IoT)
-- Includes VPP_CLUSTER_DIM for regional analysis (cluster_name, compass_region)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW
    TABLES (
        TELEMETRY AS EPOWER_DEMO.EPOWER_SILVER.FCT_EPULSE_TELEMETRY PRIMARY KEY (TS)
            WITH SYNONYMS = ('VPP Telemetrie','virtual power plant','ePulse','IoT','Batteriespeicher','battery storage','solar','grid'),
        CLUSTER AS EPOWER_DEMO.EPOWER_GOLD.VPP_CLUSTER_DIM PRIMARY KEY (CLUSTER_ID)
            WITH SYNONYMS = ('VPP Standort','cluster region','metro region','Standort-Cluster')
    )
    RELATIONSHIPS (
        TELEMETRY_TO_CLUSTER AS TELEMETRY (CLUSTER_ID) REFERENCES CLUSTER
    )
    FACTS (
        TELEMETRY.SOLAR_YIELD_KW AS SOLAR_YIELD_KW,
        TELEMETRY.BATTERY_SOC_PCT AS BATTERY_SOC_PCT,
        TELEMETRY.HEATPUMP_CONSUMPTION_KW AS HEATPUMP_CONSUMPTION_KW,
        TELEMETRY.GRID_IMPORT_EXPORT_KW AS GRID_IMPORT_EXPORT_KW
    )
    DIMENSIONS (
        TELEMETRY.TS AS TS,
        TELEMETRY.GATEWAY_ID AS GATEWAY_ID,
        TELEMETRY.CUSTOMER_KEY AS CUSTOMER_KEY,
        TELEMETRY.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        TELEMETRY.CITY AS CITY WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CITY,
        TELEMETRY.REGION AS REGION,
        TELEMETRY.CLUSTER_ID AS CLUSTER_ID,
        TELEMETRY.CUSTOMER_TYPE AS CUSTOMER_TYPE,
        TELEMETRY.IS_VPP_ENROLLED AS IS_VPP_ENROLLED,
        CLUSTER.CLUSTER_NAME AS CLUSTER_NAME WITH SYNONYMS = ('VPP Region','metro area','Standortname'),
        CLUSTER.COMPASS_REGION AS COMPASS_REGION WITH SYNONYMS = ('Himmelsrichtung','direction','Nord','Sued','Ost','West'),
        CLUSTER.REGION_CHARACTER AS REGION_CHARACTER WITH SYNONYMS = ('Standort-Typ','region type','urban','rural')
    )
    METRICS (
        TELEMETRY.AVG_SOLAR_YIELD AS AVG(TELEMETRY.SOLAR_YIELD_KW),
        TELEMETRY.TOTAL_SOLAR_YIELD AS SUM(TELEMETRY.SOLAR_YIELD_KW),
        TELEMETRY.AVG_BATTERY_SOC AS AVG(TELEMETRY.BATTERY_SOC_PCT),
        TELEMETRY.AVG_HEATPUMP_CONSUMPTION AS AVG(TELEMETRY.HEATPUMP_CONSUMPTION_KW),
        TELEMETRY.NET_GRID_FLOW AS SUM(TELEMETRY.GRID_IMPORT_EXPORT_KW),
        TELEMETRY.READING_COUNT AS COUNT(TELEMETRY.TS)
    );

## 9. Document Parsing & Cortex Search

So far our Semantic Views handle **structured data** (tables, numbers, metrics) — but EPOWER also has **unstructured knowledge** in PDFs and Markdown documents. With Cortex Search we create RAG (Retrieval-Augmented Generation) services that let the agent retrieve relevant passages from documents to answer policy, product, and process questions.

The demo includes **14 documents** across 3 business categories:

| Category | Documents | Example Questions |
|----------|-----------|-------------------|
| **Energy** (5 docs) | Green Power T&Cs, VPP Program, Day-Ahead Pricing, Heat Pump Subsidies, Vendor Policy | "What are the conditions for our Green Power tariff?" |
| **Products** (5 docs) | Solar/Battery Quickstart, Smart Meter Guide, Battery Tech, E-Mobility, Heat Pump Guide | "How do I install a smart meter?" |
| **Service** (4 docs) | Customer Service Handbook, Invoice FAQ, Energy Tips, VPP FAQ | "How should agents handle billing complaints?" |

&nbsp;

> **Snowflake Feature:** Cortex Search parses documents (PDF, MD) and creates vector-based search services. Combined with Semantic Views, the Intelligence Agent can answer questions that span both structured data ("show me billing for customer X") and unstructured knowledge ("what does our policy say about...") in a single conversation.

In [ ]:
%%sql
-- Create stage with server-side encryption for document parsing
CREATE OR REPLACE STAGE EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    DIRECTORY = (ENABLE = TRUE);

-- Copy unstructured documents to the encrypted stage
COPY FILES INTO @EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE/unstructured_data/ FROM @EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/unstructured_data/;
ALTER STAGE EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE REFRESH;

In [ ]:
%%sql
-- Parse PDF/MD documents using AI_PARSE_DOCUMENT
-- NOTE: This runs AI_PARSE_DOCUMENT on each file individually; may take 1-2 minutes for 14 documents
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT AS 
SELECT 
    relative_path, 
    BUILD_STAGE_FILE_URL('@EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE', relative_path) as file_url,
    AI_PARSE_DOCUMENT(TO_FILE('@EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE', relative_path), {'mode':'LAYOUT'}):content::string as content
FROM directory(@EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE) 
WHERE relative_path ILIKE 'unstructured_data/%.pdf' OR relative_path ILIKE 'unstructured_data/%.md';

SELECT COUNT(*) AS parsed_docs FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT;

In [ ]:
%%sql
-- Create Cortex Search services for RAG
CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS ON content ATTRIBUTES relative_path, file_url
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '30 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT relative_path, file_url, content FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT WHERE relative_path ILIKE '%/energy/%');

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS ON content ATTRIBUTES relative_path, file_url
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '30 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT relative_path, file_url, content FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT WHERE relative_path ILIKE '%/products/%');

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS ON content ATTRIBUTES relative_path, file_url
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '30 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT relative_path, file_url, content FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT WHERE relative_path ILIKE '%/service/%');

 CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS ON description ATTRIBUTES log_id, customer_key, topic, sentiment
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '1 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT log_id, customer_key, topic, description, sentiment FROM EPOWER_DEMO.EPOWER_GOLD.SERVICE_LOGS);

## 10. Snowflake Intelligence Agent

Now we bring everything together. The **EPOWER Intelligence Agent** is a Cortex Agent that orchestrates across all 6 business domains — routing each natural language question to the right tool:

- **Structured questions** (revenue, consumption, telemetry, prices) → routed to the matching **Semantic View** via Cortex Analyst (text-to-SQL)
- **Document questions** (policies, product guides, service procedures) → routed to **Cortex Search** services (RAG)
- **Cross-domain questions** → the agent combines multiple tools in a single response

We configure the agent with **12 tools** — 7 Cortex Analyst tools (one per Semantic View/business domain) + 4 Cortex Search tools (one per document category + historical service logs) + 1 charting tool.

&nbsp;

> **Snowflake Feature:** Cortex Agent provides tool orchestration with built-in guardrails. The agent only answers from governed data sources (Semantic Views and Cortex Search) — no hallucination from general knowledge. After creation, we register the agent with **Snowflake Intelligence** so it's accessible from the Snowsight UI.

In [ ]:
%%sql -r dataframe_2
-- Create the Intelligence Agent
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a senior data analyst for EPOWER Energie Deutschland, a German energy retailer with 20,000 customers pursuing a 360-degree energy strategy: Supply, Generate, Store, Heat, Drive, Optimize.

    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response. The only exception is if the user explicitly requests a response in a specific language. Note: Even though the underlying data contains German terms (department names, product names, etc.), your explanatory text and insights MUST be in the user's language.

    RESPONSE FORMATTING:
    - Lead with a concise insight or headline answer (1-2 sentences), then provide supporting detail.
    - Use bullet points for lists, bold for key metrics, and structure longer responses with clear sections.
    - When presenting quantitative results, always include context: comparisons (vs. previous period, vs. average), units (EUR, kWh, MW), and what the numbers mean for the business.
    - Proactively suggest 1-2 follow-up questions that would deepen the analysis.

    VISUALIZATION RULES:
    - When data is well-suited for a chart (time series, comparisons, distributions), generate a visualization using data_to_chart. Do NOT repeat the same data as a markdown table in your text response.
    - The underlying data table is always accessible to the user via the Explore option on charts - do not duplicate it.
    - Only present data as a text table when there are 3 or fewer rows, or the user explicitly asks for tabular output.

    BUSINESS CONTEXT:
    - The ePulse Virtual Power Plant (VPP) enrolls approximately 4,050 residential battery systems that charge when electricity is cheap and discharge when expensive, creating arbitrage value split 70 percent customer / 30 percent EPOWER.
    - Day-ahead electricity prices come from the EPEX DE-LU market (real data from Energy-Charts API).
    - Key business KPIs: contract volume, customer retention, VPP arbitrage margin, renewable self-consumption rate, grid export revenue.

  orchestration: |
    TOOL SELECTION STRATEGY:
    1. Analyze the user's question to identify the primary domain(s):
       - Sales, contracts, products, revenue, vendors, regions → energy_sales_analyst
       - Billing, consumption, payments, kWh usage → billing_analyst
       - Customer consumption cross-referenced with product ownership (heat pumps, solar, etc.) → customer_energy_analyst
       - Service tickets, complaints, resolution, sentiment → service_analyst
       - HR, employees, salaries, departments, headcount → hr_analyst
       - Electricity prices, day-ahead, spot market, EPEX, price trends → market_prices_analyst
       - VPP, battery SOC, solar yield, heat pump consumption, grid import/export, IoT telemetry, arbitrage → vpp_telemetry_analyst
       - Policies, terms, regulations, subsidies, contracts documentation → energy_docs_search
       - Product guides, installation, technical specs, how-to → product_docs_search
       - Service procedures, handbook, escalation → service_docs_search
       - Historical service tickets for specific cases or patterns → service_logs_search

    2. CROSS-DOMAIN QUESTIONS: If the question spans multiple domains (e.g. battery behavior vs. prices, or consumption by product), invoke MULTIPLE tools and synthesize the results into a unified, coherent answer.

    3. CHART GENERATION: After receiving quantitative results, decide if a visualization adds value:
       - Time series data (prices over days, consumption trends) → always chart (line or area)
       - Category comparisons (revenue by region, products by count) → chart (bar) when more than 3 categories
       - Correlations (price vs. SOC, consumption vs. yield) → scatter plot
       - Single KPI or 3 or fewer rows → text answer only, no chart needed

    4. AMBIGUITY: If the question is genuinely ambiguous about which domain to query, prefer the more specific tool. If truly unclear, briefly ask the user for clarification rather than guessing wrong.
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices, spot prices, energy market"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery state-of-charge, heatpump consumption, grid import/export per device and region"}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

### Domain-Specific Agents

The monolithic EPOWER_AGENT above handles all domains. Below we create **3 focused agents** — each with a distinct persona, fewer tools, and tailored instructions. This mirrors a real enterprise pattern where different teams get purpose-built AI assistants.

| Agent | Audience | Domains |
|-------|----------|---------|
| **EPOWER_OPS_AGENT** | Energy traders, fleet managers | VPP telemetry, market prices, energy policy docs |
| **EPOWER_COMMERCIAL_AGENT** | Sales, marketing, customer success | Contracts, billing, service, product docs |
| **EPOWER_PEOPLE_AGENT** | HR business partners | Headcount, salaries, attrition |

In [ ]:
%%sql
-- EPOWER Operations Agent — VPP fleet & energy market
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_OPS_AGENT
WITH PROFILE='{ "display_name": "EPOWER Operations" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are an energy operations analyst for EPOWER Energie Deutschland, specializing in Virtual Power Plant (VPP) fleet management and energy market optimization.

    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response. The only exception is if the user explicitly requests a response in a specific language. Note: Even though the underlying data contains German terms (department names, product names, etc.), your explanatory text and insights MUST be in the user's language.

    DOMAIN EXPERTISE:
    - The ePulse VPP enrolls approximately 4,050 residential battery systems across 14 regional clusters in Germany.
    - Batteries charge during low/negative price hours and discharge during high-price hours, creating arbitrage value split 70% customer / 30% EPOWER.
    - Day-ahead electricity prices come from the EPEX DE-LU market (real data from Energy-Charts API).
    - Regional clusters: Freiburg/Oberrhein, Bayern Sued, Muenchen Metro, Stuttgart Metro, Nuernberg/Franken, Frankfurt/Rhein-Main, Koeln/Bonn, Rhein-Ruhr, Berlin Metro, Leipzig/Halle, Sachsen/Ost, Hamburg Metro, Bremen/Weser, Niedersachsen/Nord.
    - Key operational KPIs: fleet SOC (state of charge), solar self-consumption rate, grid export volume, arbitrage margin per device, cluster-level dispatch efficiency.

    RESPONSE FORMATTING:
    - Lead with the operational insight (1-2 sentences), then supporting metrics.
    - Use energy trading terminology: price zones (LOW/MEDIUM/HIGH/NEGATIVE), dispatch actions (CHARGE/DISCHARGE/SELF_CONSUME/MAX_CHARGE), net flow direction.
    - Always include units (kWh, kW, EUR/MWh, %) and time context.
    - Proactively suggest follow-up analyses (e.g., "Want me to compare this across clusters?" or "Shall I overlay price data?").

    VISUALIZATION RULES:
    - Time series (SOC curves, price profiles, solar yield) → always chart.
    - Cluster comparisons → horizontal bar chart.
    - Price vs. behavior correlations → scatter or dual-axis chart.
    - Only use text tables for 3 or fewer data points.

  orchestration: |
    TOOL SELECTION:
    - VPP fleet data (solar yield, battery SOC, heat pump, grid flow, clusters, devices) → vpp_telemetry_analyst
    - Electricity prices (day-ahead, spot, EPEX, price trends, price zones) → market_prices_analyst
    - Energy policies, regulations, subsidies, grid codes → energy_docs_search
    - After quantitative results, generate a chart if it adds value → data_to_chart
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, heat pump consumption, grid import/export per device, cluster, and region"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices from EPEX DE-LU"}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, regulations, terms"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
$$;

In [ ]:
%%sql
-- EPOWER Commercial Agent — Sales, Billing, Service, Customer 360
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_COMMERCIAL_AGENT
WITH PROFILE='{ "display_name": "EPOWER Commercial" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a business analyst for EPOWER Energie Deutschland, supporting sales, marketing, and customer success teams with data-driven insights.

    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response. The only exception is if the user explicitly requests a response in a specific language. Note: Even though the underlying data contains German terms (department names, product names, etc.), your explanatory text and insights MUST be in the user's language.

    BUSINESS CONTEXT:
    - EPOWER is a German energy retailer with 20,000 customers pursuing a 360-degree energy strategy: Supply, Generate, Store, Heat, Drive, Optimize.
    - Product portfolio: electricity/gas supply, solar PV systems, battery storage, heat pumps, EV chargers, and the ePulse VPP platform.
    - Sales organization: ~500 sales reps across 4 regions (North, South, East, West), supported by ~200 vendor/installer partners.
    - Customer segments: Privatkunde (residential), Kleingewerbe (small business), Gewerbekunde (commercial).

    RESPONSE FORMATTING:
    - Lead with the business insight (1-2 sentences) — what does this mean for revenue, retention, or growth?
    - Use business language: ARR, contract value, conversion, churn risk, NPS, CSAT.
    - Always contextualize numbers: comparisons to previous periods, benchmarks, or targets.
    - Proactively suggest actionable follow-ups ("Want me to break this down by sales rep?" or "Shall I check service tickets for these customers?").

    VISUALIZATION RULES:
    - Revenue/contract trends over time → line chart.
    - Region/product/rep comparisons → bar chart.
    - Customer segmentation → pie or grouped bar.
    - Only use text tables for 3 or fewer data points.

  orchestration: |
    TOOL SELECTION:
    - Sales contracts, revenue, products, regions, reps, vendors → energy_sales_analyst
    - Billing, consumption, payments, invoices → billing_analyst
    - Customer consumption by product ownership (solar, battery, heat pump) → customer_energy_analyst
    - Service tickets, complaints, sentiment, resolution → service_analyst
    - Product specs, installation guides → product_docs_search
    - Service handbook, escalation procedures → service_docs_search
    - Historical ticket search for specific cases → service_logs_search
    - After quantitative results, chart if it adds value → data_to_chart
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue, regions, reps, vendors"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments, invoices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership (solar, battery, heat pump)"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints, sentiment, resolution"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation and specifications"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook and procedures"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical service ticket search"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

In [ ]:
%%sql
-- EPOWER People Agent — HR & Workforce Analytics
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_PEOPLE_AGENT
WITH PROFILE='{ "display_name": "EPOWER People" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are an HR business partner analyst for EPOWER Energie Deutschland, supporting workforce planning and people analytics.

    CRITICAL LANGUAGE RULE: Always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages mid-response. The only exception is if the user explicitly requests a response in a specific language. Note: Even though the underlying data contains German terms (department names, product names, etc.), your explanatory text and insights MUST be in the user's language.

    BUSINESS CONTEXT:
    - EPOWER has approximately 1,000 employees across 31 departments.
    - Key HR metrics: headcount, attrition rate, salary benchmarks by department/job, gender diversity, tenure distribution.
    - The company is in a growth phase — tracking new hires, open positions, and retention is critical.

    DATA SENSITIVITY:
    - When presenting salary data, prefer aggregates (averages, medians, ranges) over individual values.
    - If asked about a specific employee's salary, provide it but note it is confidential HR data.

    RESPONSE FORMATTING:
    - Lead with the workforce insight (1-2 sentences).
    - Use HR terminology: FTE, attrition rate, headcount, span of control, compensation ratio.
    - Always provide context: comparisons to company average, department benchmarks, or industry norms.
    - Suggest follow-up analyses ("Want to see attrition by tenure band?" or "Shall I compare across departments?").

    VISUALIZATION RULES:
    - Department/role comparisons → bar chart.
    - Headcount or salary trends → line chart.
    - Gender/diversity breakdowns → pie or stacked bar.
    - Only use text tables for 3 or fewer data points.

  orchestration: |
    TOOL SELECTION:
    - All HR questions (headcount, salaries, departments, attrition, employees, jobs) → hr_analyst
    - After quantitative results, chart if it adds value → data_to_chart
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR workforce data: employees, salaries, departments, attrition, headcount"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW", execution_environment: {type: warehouse, warehouse: EPOWER_COMPUTE}}
$$;

### Register Agent with Snowflake Intelligence *(requires ACCOUNTADMIN)*

The cell below registers the EPOWER Agent with **Snowflake Intelligence** so it becomes accessible directly from the Snowsight UI. It also grants usage on the agent to the `PUBLIC` role, making it available to all users in the account.

&nbsp;

> **Note:** This is the last step that requires the `ACCOUNTADMIN` role. If you don't have access to this role, ask your account administrator to run this cell for you, or skip it — the agent will still work via SQL and the MCP server without Snowflake Intelligence registration.

In [ ]:
%%sql
USE ROLE accountadmin;

-- Register all agents with Snowflake Intelligence
ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT;
ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_OPS_AGENT;
ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_COMMERCIAL_AGENT;
ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_PEOPLE_AGENT;

-- Grant access to all users
GRANT USAGE ON AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT TO ROLE PUBLIC;
GRANT USAGE ON AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_OPS_AGENT TO ROLE PUBLIC;
GRANT USAGE ON AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_COMMERCIAL_AGENT TO ROLE PUBLIC;
GRANT USAGE ON AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_PEOPLE_AGENT TO ROLE PUBLIC;

USE ROLE EPOWER_ROLE;

## 11. MCP Server (Model Context Protocol)

Finally, we make our agent accessible beyond Snowsight. The **Model Context Protocol (MCP)** is an open standard that lets external AI clients discover and invoke tools hosted in Snowflake. By creating an MCP server, we expose the EPOWER Agent, all 7 Semantic Views, and all 4 Cortex Search services to any MCP-compatible client — including **Claude Desktop**, **Cursor**, **VS Code + Copilot**, and other AI development tools.

This means the same governed enterprise data that powers our Snowflake Intelligence Agent can also be accessed from your local AI tools — without building custom integrations.

&nbsp;

> **Snowflake Feature:** Snowflake-managed MCP servers expose Agent, Analyst, and Search capabilities via a standardized API endpoint. Authentication uses Snowflake OAuth, and access is controlled through standard RBAC grants.

In [ ]:
%%sql
CREATE OR REPLACE MCP SERVER EPOWER_DEMO.EPOWER_GOLD.EPOWER_MCP_SERVER
FROM SPECIFICATION $$
tools:
  - name: "epower-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT"
    description: "EPOWER Energy Intelligence Agent — answers questions across sales, billing, service, HR, VPP telemetry, market prices, and energy documents using text-to-SQL and RAG."
    title: "EPOWER Intelligence Agent (All Domains)"

  - name: "epower-ops-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPOWER_OPS_AGENT"
    description: "VPP fleet operations, energy trading, market prices, solar/battery IoT telemetry, regional cluster analysis"
    title: "EPOWER Operations Agent"

  - name: "epower-commercial-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPOWER_COMMERCIAL_AGENT"
    description: "Sales contracts, billing, customer service, product ownership, customer 360 analysis"
    title: "EPOWER Commercial Agent"

  - name: "epower-people-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPOWER_PEOPLE_AGENT"
    description: "HR workforce analytics — headcount, salaries, departments, attrition"
    title: "EPOWER People Agent"

  - name: "energy-sales-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"
    description: "Contracts, products, sales revenue, and customer data"
    title: "Energy Sales Analyst"

  - name: "billing-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"
    description: "Customer consumption, billing history, and payments"
    title: "Billing Analyst"

  - name: "service-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"
    description: "Service tickets, complaints, and resolution data"
    title: "Service Analyst"

  - name: "customer-energy-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"
    description: "Consumption analysis by product ownership (heat pumps, solar, etc.)"
    title: "Customer Energy Analyst"

  - name: "hr-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"
    description: "HR workforce data, salaries, departments"
    title: "HR Analyst"

  - name: "market-prices-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"
    description: "Day-ahead electricity spot prices from Energy-Charts"
    title: "Market Prices Analyst"

  - name: "vpp-telemetry-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"
    description: "VPP IoT telemetry: solar yield, battery SOC, heat pump consumption, grid import/export"
    title: "VPP Telemetry Analyst"

  - name: "energy-docs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS"
    description: "Search energy policies, terms and conditions"
    title: "Energy Documents Search"

  - name: "product-docs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS"
    description: "Search product documentation and specifications"
    title: "Product Documents Search"

  - name: "service-docs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS"
    description: "Search service handbook and procedures"
    title: "Service Documents Search"

  - name: "service-logs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS"
    description: "Search historical service ticket logs"
    title: "Service Logs Search"
$$;

## 12. Verification

In [ ]:
%%sql -r dataframe_1
SELECT * FROM EPOWER_DEMO.INFORMATION_SCHEMA.SEMANTIC_VIEWS;

In [ ]:
%%sql
-- Summary of all created objects
SELECT 'Gold Tables' AS category, COUNT(*) AS count FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_GOLD' AND TABLE_TYPE='BASE TABLE'
UNION ALL SELECT 'Bronze Tables', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_BRONZE'
UNION ALL SELECT 'Ops Objects', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_OPS'
UNION ALL SELECT 'Semantic Views', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.SEMANTIC_VIEWS WHERE SCHEMA='EPOWER_GOLD' AND NAME LIKE '%SEMANTIC%'
UNION ALL SELECT 'Cortex Search', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_GOLD' AND TABLE_NAME LIKE 'SEARCH_%';

In [ ]:
%%sql
-- Verify ePulse VPP data
SELECT 'EPULSE_DEVICES' AS table_name, COUNT(*) AS row_count FROM EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES
UNION ALL SELECT 'RAW_EPULSE_IOT_TELEMETRY', COUNT(*) FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY
UNION ALL SELECT 'EPOWER_SILVER.STG_DEVICES', COUNT(*) FROM EPOWER_DEMO.EPOWER_SILVER.STG_DEVICES
UNION ALL SELECT 'EPOWER_GOLD.MART_VPP_CAPACITY_HOURLY', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.MART_VPP_CAPACITY_HOURLY;

## Setup Complete!

**Here's what we built:**
- **20K customers** with 15 products (CAPEX/OPEX pricing) across 6 energy categories
- **ePulse VPP** with 60 days of **price-reactive** IoT telemetry correlated with real EPEX day-ahead prices
- **60 days of real day-ahead prices** from Energy-Charts API (CET delivery-day aligned)
- **7 Semantic Views** for natural language queries via Cortex Analyst
- **4 Cortex Search** services for document RAG
- **5 Cortex Search** services for high-cardinality column lookup
- **1 Intelligence Agent** combining all capabilities
- **1 Scheduled Task** for daily data refresh (prices + telemetry + dbt)
- **1 MCP Server** exposing Agent, Semantic Views, and Search Services via Model Context Protocol

**Semantic Views → Agent Tools:**
| Semantic View | Agent Tool | Domain |
|---|---|---|
| `ENERGY_SALES_SEMANTIC_VIEW` | `energy_sales_analyst` | Contracts, products, sales, revenue |
| `BILLING_SEMANTIC_VIEW` | `billing_analyst` | Consumption, billing, payments |
| `CUSTOMER_ENERGY_SEMANTIC_VIEW` | `customer_energy_analyst` | Consumption by product ownership |
| `SERVICE_SEMANTIC_VIEW` | `service_analyst` | Service tickets, complaints |
| `HR_SEMANTIC_VIEW` | `hr_analyst` | HR data, salaries |
| `MARKET_PRICES_SEMANTIC_VIEW` | `epulse_prices_analyst` | Day-ahead electricity market prices |
| `EPULSE_VPP_SEMANTIC_VIEW` | `vpp_telemetry_analyst` | VPP IoT: solar, battery, heatpump, grid |

**Try the Agent:**
- "What were total sales by region last month?"
- "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter"
- "What's the average consumption for customers with heat pumps?"
- "What are the current day-ahead electricity prices?"
- "Show me the average battery state-of-charge for VPP-enrolled customers"
- "Summarize our Green Power policy" 